In [ ]:
import pandas as pd
import numpy as np
from src.oracle_db_connection import get_oracle_connection

In [ ]:
import os

import boto3
from botocore.client import Config
from dotenv import load_dotenv


load_dotenv()

BUCKET_NAME = os.environ["MINIO_BUCKET"]

s3_client = boto3.client(
    "s3",
    endpoint_url=os.environ["MINIO_ENDPOINT"],
    aws_access_key_id=os.environ["MINIO_ACCESS_KEY"],
    aws_secret_access_key=os.environ["MINIO_SECRET_KEY"],
    region_name="us-east-1",
    config=Config(signature_version="s3v4"),
)

In [ ]:
from io import BytesIO
from datetime import datetime, timezone

tables = ["CUSTOMERS","CLAIMS","POLICIES"]


for table in tables:
    with get_oracle_connection() as conn:

        # -------------------------
        # 1. Get last watermark
        # -------------------------

        watermark_df = pd.read_sql(
            """
            SELECT LAST_WATERMARK
            FROM ETL_METADATA
            WHERE TABLE_NAME = :table_name
            """,
            conn,
            params={"table_name": table}
        )

        last_watermark = watermark_df.iloc[0]["LAST_WATERMARK"]

        print("Previous watermark:", last_watermark)


        # -------------------------
        # 2. Extract incrementally
        # -------------------------

        df = pd.read_sql(
            f"""
            SELECT *
            FROM {table}
            WHERE LAST_UPDATED > :watermark
            """,
            conn,
            params={"watermark": last_watermark}
        )

        print(f"Extracted {len(df)} rows")


        # -------------------------
        # 3. Write the data
        # -------------------------

        if not df.empty:

            parquet_buffer = BytesIO()

            df.to_parquet(
                parquet_buffer,
                index=False,
            )

            load_timestamp = datetime.now(timezone.utc)
            parquet_buffer.seek(0)
            object_key = (
            f"raw/{table}/"
            f"load_date={load_timestamp:%Y-%m-%d}/"
            f"{table}_{load_timestamp:%Y%m%dT%H%M%S%fZ}.parquet"
            )

            s3_client.put_object(
                Bucket=BUCKET_NAME,
                Key=object_key,
                Body=parquet_buffer.getvalue(),
                ContentType="application/octet-stream",
            )

            print(f"Uploaded to MinIO: {object_key}")

            # -------------------------
            # 4. Update watermark
            # -------------------------

            new_watermark = df["LAST_UPDATED"].max()

            conn.cursor().execute(
                """
                UPDATE ETL_METADATA
                SET LAST_WATERMARK = :new_watermark
                WHERE TABLE_NAME = :table_name
                """,
                {
                    "new_watermark": new_watermark,
                    "table_name": table
                }
            )

            conn.commit()

            print("New watermark:", new_watermark)

        else:

            print("No new data.")

In [ ]:
from io import BytesIO
from datetime import datetime, timezone

tables = ["AGENTS","CLAIM_PAYMENTS","INSURANCE_PRODUCTS","POLICY_COVERAGES","POLICY_STATUS_HISTORY","PREMIUM_TRANSACTIONS"]


for table in tables:
    with get_oracle_connection() as conn:

        # -------------------------
        # 1. Get last watermark
        # -------------------------

        watermark_df = pd.read_sql(
            """
            SELECT LAST_WATERMARK
            FROM ETL_METADATA
            WHERE TABLE_NAME = :table_name
            """,
            conn,
            params={"table_name": table}
        )

        last_watermark = watermark_df.iloc[0]["LAST_WATERMARK"]

        print("Previous watermark:", last_watermark)


        # -------------------------
        # 2. Extract incrementally
        # -------------------------

        df = pd.read_sql(
            f"""
            SELECT *
            FROM {table}
            WHERE LAST_UPDATED > :watermark
            """,
            conn,
            params={"watermark": last_watermark}
        )

        print(f"Extracted {len(df)} rows")


        # -------------------------
        # 3. Write the data
        # -------------------------

        if not df.empty:

            parquet_buffer = BytesIO()

            df.to_parquet(
                parquet_buffer,
                index=False,
            )

            load_timestamp = datetime.now(timezone.utc)
            parquet_buffer.seek(0)
            object_key = (
            f"raw/{table}/"
            f"load_date={load_timestamp:%Y-%m-%d}/"
            f"{table}_{load_timestamp:%Y%m%dT%H%M%S%fZ}.parquet"
            )

            s3_client.put_object(
                Bucket=BUCKET_NAME,
                Key=object_key,
                Body=parquet_buffer.getvalue(),
                ContentType="application/octet-stream",
            )

            print(f"Uploaded to MinIO: {object_key}")

            # -------------------------
            # 4. Update watermark
            # -------------------------

            new_watermark = df["LAST_UPDATED"].max()

            conn.cursor().execute(
                """
                UPDATE ETL_METADATA
                SET LAST_WATERMARK = :new_watermark
                WHERE TABLE_NAME = :table_name
                """,
                {
                    "new_watermark": new_watermark,
                    "table_name": table
                }
            )

            conn.commit()

            print("New watermark:", new_watermark)

        else:

            print("No new data.")